In [ ]:
import torch
import math
import random
from torch import nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from torchvision import transforms
import torchvision.transforms as T
from torch.utils.data import Dataset
from torchvision.utils import make_grid
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from PIL import Image 
import numpy as np
import os
from math import log10
from skimage.metrics import structural_similarity as ssim

In [ ]:
class MRICTDataset(Dataset):
    def __init__(self, root_dir, transform = None, mode = 'Train'):
        self.root_dir = root_dir
        self.mri_dir = os.listdir(root_dir + '/' + mode + "_MRI")
        self.ct_dir = os.listdir(root_dir + '/' + mode + "_CT")
        self.transform = transform
        self.mode = mode

    def __len__(self):
        return len(self.mri_dir)

    def __getitem__(self, idx):
        mri_path = './Dataset/' + self.mode + '_MRI/' + self.mri_dir[idx]
        ct_path = './Dataset/' + self.mode + '_CT/' +  self.ct_dir[idx]
        mri = Image.open(mri_path)
        ct = Image.open(ct_path)
        mri = self.transform(mri)
        ct = self.transform(ct)
        return mri, ct

In [ ]:
epochs = 500
display_step = 50
learning_rate = 0.0002
beta_1 = 0.9
beta_2 = 0.999
lambda_ = 1.0
image_size = 256
device = 'cuda'
trained = False
batch_size = 4
epoch_to_load = 0

In [ ]:
stats = (0.5), (0.5)

In [ ]:
transform = T.Compose([
    T.Grayscale(num_output_channels = 1),
    T.Resize(image_size),
    T.CenterCrop(image_size),
    T.ToTensor(),
    T.Normalize(*stats)
])

In [ ]:
train_ds = MRICTDataset('./Dataset', transform = transform, mode = 'Train')
test_ds = MRICTDataset('./Dataset', transform = transform, mode = 'Test')
train_loader = DataLoader(train_ds, batch_size, shuffle = True, num_workers = 0, pin_memory = True)
test_loader = DataLoader(test_ds, batch_size, shuffle = True, num_workers = 0, pin_memory = True)

In [ ]:
def show_tensor_images(image_tensor, num_images = 25, size = (1, 28, 28)):
    image_tensor = (image_tensor + 1) / 2
    image_unflat = image_tensor.detach().cpu()
    image_grid = make_grid(image_unflat[:num_images], nrow = 5)
    images = image_unflat[:num_images]
    plt.imshow(image_grid.permute(1, 2, 0).squeeze())
    plt.show()
    for i in range(images.shape[0]):
        image = images[i].squeeze().numpy()
        image = (image * 255).astype(np.uint8)
        image = Image.fromarray(image, mode = 'L')
        image.save('Generated_Images/image_' + str(i) + '.png')

In [ ]:
def generate_noise(n_examples, z_dim, device = 'cpu'):
    noise = torch.randn(n_examples, z_dim, device = device)
    return noise

In [ ]:
def weights_init(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
        torch.nn.init.normal_(m.weight, 0.0, 0.02)
    if isinstance(m, nn.BatchNorm2d):
        torch.nn.init.normal_(m.weight, 0.0, 0.02)
        torch.nn.init.constant_(m.bias, 0)

In [ ]:
class ResNetBlock(nn.Module):
    def __init__(self, dim):
        super(ResNetBlock, self).__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, kernel_size = 3, stride = 1, padding = 0),
            nn.InstanceNorm2d(dim),
            nn.ReLU(inplace = True),
            nn.Dropout(0.3),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, kernel_size = 3, stride = 1, padding = 0),
            nn.InstanceNorm2d(dim)
        )
    def forward(self, x):
        return x + self.block(x)

In [ ]:
class Generator(nn.Module):
    def __init__(self, input_channels = 1, output_channels = 1, noise_dim = 8, resnet_blocks = 9):
        super(Generator, self).__init__()
        self.in_channels = input_channels + noise_dim
        self.down1 = nn.Sequential(
            nn.ReflectionPad2d(2),
            nn.Conv2d(self.in_channels, 64, kernel_size = 5, stride = 1, padding = 0),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace = True)
        )
        self.down2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size = 4, stride = 2, padding = 1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace = True)
        )
        self.down3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size = 4, stride = 2, padding = 1),
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace = True)
        )
        self.res_blocks = nn.Sequential(*[ResNetBlock(256) for _ in range(resnet_blocks)])
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size = 4, stride = 2, padding = 1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace = True)
        )
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(256, 64, kernel_size = 4, stride = 2, padding = 1),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace = True)
        )
        self.up3 = nn.Sequential(
            nn.ReflectionPad2d(2),
            nn.Conv2d(128, output_channels, kernel_size = 5, stride = 1, padding = 0),
            nn.Tanh()
        )
    def forward(self, image):
        d1 = self.down1(image)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        r = self.res_blocks(d3)
        u1 = self.up1(r)
        u1 = torch.cat([u1, d2], 1)
        u2 = self.up2(u1)
        u2 = torch.cat([u2, d1], 1)
        out = self.up3(u2)
        return out

In [ ]:
class Critic(nn.Module):
    def __init__(self, mri_dim = 1, ct_dim = 1):
        super(Critic, self).__init__()
        self.in_channels = mri_dim + ct_dim
        self.block1 = nn.Sequential(
            nn.Conv2d(self.in_channels, 64, kernel_size = 4, stride = 2, padding = 1),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace = True)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size = 4, stride = 2, padding = 1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace = True)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size = 4, stride = 2, padding = 1),
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace = True)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size = 4, stride = 1, padding = 1),
            nn.InstanceNorm2d(512),
            nn.ReLU(inplace = True)
        )
        self.block5 = nn.Conv2d(512, 1, kernel_size = 4, stride = 1, padding = 1)
    def forward(self, image):
        x = self.block1(image)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        out = self.block5(x)
        return out

In [ ]:
def denormalize_to_01(tensor):
    # Convert [-1, 1] → [0, 1]
    return (tensor + 1) / 2

def compute_psnr(fake, real):
    fake_dn = denormalize_to_01(fake)
    real_dn = denormalize_to_01(real)
    mse = F.mse_loss(fake_dn, real_dn)
    if mse == 0:
        return 100
    # Use data_range = 1 for [0, 1] scale
    return 10 * log10(1.0 / mse.item())

def compute_ssim(fake, real):
    fake_dn = denormalize_to_01(fake)
    real_dn = denormalize_to_01(real)
    fake_np = fake_dn.detach().cpu().permute(0, 2, 3, 1).numpy()
    real_np = real_dn.detach().cpu().permute(0, 2, 3, 1).numpy()

    ssim_values = []
    for f, r in zip(fake_np, real_np):
        s = ssim(f, r, channel_axis = -1, data_range = 1.0)  # correct data_range
        ssim_values.append(s)
    return np.mean(ssim_values)

def compute_mae(fake, real):
    fake_dn = denormalize_to_01(fake)
    real_dn = denormalize_to_01(real)
    return F.l1_loss(fake_dn, real_dn).item()

def compute_mse(fake, real):
    fake_dn = denormalize_to_01(fake)
    real_dn = denormalize_to_01(real)
    return F.mse_loss(fake_dn, real_dn).item()

def compute_pcc(fake, real):
    fake_dn = denormalize_to_01(fake)
    real_dn = denormalize_to_01(real)
    fake_flat = fake_dn.detach().cpu().view(fake_dn.size(0), -1)
    real_flat = real_dn.detach().cpu().view(real_dn.size(0), -1)
    pcc = []
    for f, r in zip(fake_flat, real_flat):
        f_mean, r_mean = f.mean(), r.mean()
        num = torch.sum((f - f_mean) * (r - r_mean))
        den = torch.sqrt(torch.sum((f - f_mean) ** 2) * torch.sum((r - r_mean) ** 2))
        pcc.append((num / den).item())
    return np.mean(pcc)

In [ ]:
def score(epoch_to_load, test_loader, lambda_, device):
    gen = Generator().to(device)
    gen.load_state_dict(torch.load(str(epoch_to_load) + '_Generator_' + str(lambda_) + '.pth'))
    
    psnrs = []
    ssims = []
    maes = []
    mses = []
    pccs = []
    
    for mri, ct in tqdm(test_loader):
        mri = mri.to(device)
        ct = ct.to(device)
        cur_batch_size = len(mri)
        
        fake_noise = generate_noise(cur_batch_size, 8, device = device)
        fake_noise = fake_noise.view(cur_batch_size, 8, 1, 1)
        fake_noise = fake_noise.expand(-1, -1, 256, 256)
        gen_input = torch.cat([mri, fake_noise], 1)
        ct_synthesized = gen(gen_input)
        
        psnr_val = compute_psnr(ct_synthesized, ct)
        ssim_val = compute_ssim(ct_synthesized, ct)
        mae_val = compute_mae(ct_synthesized, ct)
        mse_val = compute_mse(ct_synthesized, ct)
        pcc_val = compute_pcc(ct_synthesized, ct)
        
        psnrs.append(psnr_val)
        ssims.append(ssim_val)
        maes.append(mae_val)
        mses.append(mse_val)
        pccs.append(pcc_val)
        
    print(f"PSNR: {np.mean(psnrs)}, SSIM: {np.mean(ssims)}, MAE: {np.mean(maes)}, MSE: {np.mean(mses)}, PCC: {np.mean(pccs)}")

In [ ]:
def GAN(trained, epochs, display_step, learning_rate, beta_1, beta_2, lambda_, dataloader, device, epoch_to_load = 0):
    gen = Generator().to(device)
    crit = Critic().to(device)
    
    gen_optimizer = torch.optim.Adam(gen.parameters(), lr = learning_rate, betas = (beta_1, beta_2))
    crit_optimizer = torch.optim.Adam(crit.parameters(), lr = learning_rate, betas = (beta_1, beta_2))
    
    if not trained:
        gen = gen.apply(weights_init)
        crit = crit.apply(weights_init)
    else:
        gen.load_state_dict(torch.load(str(epoch_to_load) + '_Generator_' + str(lambda_) + '.pth'))
        crit.load_state_dict(torch.load(str(epoch_to_load) + '_Critic_' + str(lambda_) + '.pth'))
        gen_optimizer.load_state_dict(torch.load(str(epoch_to_load) + '_GeneratorOptimizer_' + str(lambda_) + '.pth'))
        crit_optimizer.load_state_dict(torch.load(str(epoch_to_load) + '_CriticOptimizer_' + str(lambda_) + '.pth'))
        
    criterion = nn.BCEWithLogitsLoss()
    criterion_aux = nn.L1Loss()
    
    cur_step = 0
    generator_losses = []
    critic_losses = []
    
    for epoch in range(epochs):
        for mri, ct in tqdm(dataloader):
            mri = mri.to(device)
            ct = ct.to(device)
            cur_batch_size = len(mri)
            
            crit_optimizer.zero_grad()
            
            fake_noise = generate_noise(cur_batch_size, 8, device = device)
            fake_noise = fake_noise.view(cur_batch_size, 8, 1, 1)
            fake_noise = fake_noise.expand(-1, -1, 256, 256)
            gen_input = torch.cat([mri, fake_noise], 1)
            ct_synthesized = gen(gen_input)
            
            crit_input_real = torch.cat([mri, ct], 1)
            crit_input_fake = torch.cat([mri, ct_synthesized], 1)
            crit_real = crit(crit_input_real)
            crit_fake = crit(crit_input_fake)
            crit_loss_real = criterion(crit_real, torch.ones_like(crit_real))
            crit_loss_fake = criterion(crit_fake, torch.zeros_like(crit_fake))
            crit_loss = (crit_loss_real + crit_loss_fake) / 2
            critic_losses += [crit_loss.item()]
            crit_loss.backward(retain_graph = True)
            crit_optimizer.step()
            cur_step += 1
            
            gen_optimizer.zero_grad()
            adv_loss = criterion(crit_fake, torch.ones_like(crit_fake))
            l1_loss = criterion_aux(ct_synthesized, ct)
            gen_loss = adv_loss + lambda_ * l1_loss
            generator_losses += [gen_loss.item()]
            gen_loss.backward(retain_graph = True)
            gen_optimizer.step()
            
            if cur_step % display_step == 0 and cur_step > 0:
                gen_mean = sum(generator_losses[-display_step:]) / display_step
                crit_mean = sum(critic_losses[-display_step:]) / display_step
                
                psnr_val = compute_psnr(ct_synthesized, ct)
                ssim_val = compute_ssim(ct_synthesized, ct)
                mae_val = compute_mae(ct_synthesized, ct)
                mse_val = compute_mse(ct_synthesized, ct)
                pcc_val = compute_pcc(ct_synthesized, ct)

                print(f"Epoch {epoch}, Step {cur_step}: Generator loss: {gen_mean}, Critic loss: {crit_mean}, PSNR: {psnr_val}, SSIM: {ssim_val}, MAE: {mae_val}, MSE: {mse_val}, PCC: {pcc_val}")
                show_tensor_images(ct_synthesized, num_images = 10, size = (1, 256, 256))
                show_tensor_images(ct, num_images = 10, size = (1, 256, 256))
                step_bins = 20
                num_examples = (len(generator_losses) // step_bins) * step_bins
                plt.plot(
                    range(num_examples // step_bins), 
                    torch.Tensor(generator_losses[:num_examples]).view(-1, step_bins).mean(1),
                    label = "Generator Loss"
                )
                plt.plot(
                    range(num_examples // step_bins), 
                    torch.Tensor(critic_losses[:num_examples]).view(-1, step_bins).mean(1),
                    label = "Critic Loss"
                )
                plt.legend()
                plt.show()
                
                torch.save(gen.state_dict(), './' + str(epochs) + '_Generator_' + str(lambda_) + '.pth')
                torch.save(crit.state_dict(), './' + str(epochs) + '_Critic_' + str(lambda_) + '.pth')
                torch.save(gen_optimizer.state_dict(), './' + str(epochs) + '_GeneratorOptimizer_' + str(lambda_) + '.pth')
                torch.save(crit_optimizer.state_dict(), './' + str(epochs) + '_CriticOptimizer_' + str(lambda_) + '.pth')

In [ ]:
GAN(trained, epochs, display_step, learning_rate, beta_1, beta_2, lambda_, train_loader, device, epoch_to_load)

In [ ]:
GAN(trained, epochs, display_step, learning_rate, beta_1, beta_2, lambda_, train_loader, device, epoch_to_load)

In [ ]:
GAN(trained, epochs, display_step, learning_rate, beta_1, beta_2, lambda_, train_loader, device, epoch_to_load)

In [ ]:
GAN(trained, epochs, display_step, learning_rate, beta_1, beta_2, lambda_, train_loader, device, epoch_to_load)

In [ ]:
GAN(trained, epochs, display_step, learning_rate, beta_1, beta_2, lambda_, train_loader, device, epoch_to_load)

In [ ]:
GAN(trained, epochs, display_step, learning_rate, beta_1, beta_2, lambda_, train_loader, device, epoch_to_load)

## Test and Get the Stats

## Lambda_ = 1.0

In [ ]:
score(500, test_loader, 1.0, device)

In [ ]:
score(500, train_loader, 1.0, device)

## Lambda_ = 0.9

In [ ]:
score(500, test_loader, 0.9, device)

In [ ]:
score(500, train_loader, 0.9, device)

## Lambda_ = 0.8

In [ ]:
score(500, test_loader, 0.8, device)

In [ ]:
score(500, train_loader, 0.8, device)

## Lambda_ = 0.7

In [ ]:
score(500, test_loader, 0.7, device)

In [ ]:
score(500, train_loader, 0.7, device)

## Lambda_ = 0.6

In [ ]:
score(500, test_loader, 0.6, device)

In [ ]:
score(500, train_loader, 0.6, device)

## Lambda_ = 0.5

In [ ]:
score(500, test_loader, 0.5, device)

In [ ]:
score(500, train_loader, 0.5, device)

## Generate a batch of 4 images, display them, save them, and individually open them

In [ ]:
gen = Generator().to(device)
gen.load_state_dict(torch.load(str(500) + '_Generator_' + str(lambda_) + '.pth'))

In [ ]:
for mri, ct in train_loader:
    break

In [ ]:
fake_noise = generate_noise(4, 8, device = device)
fake_noise = fake_noise.view(4, 8, 1, 1)
fake_noise = fake_noise.expand(-1, -1, 256, 256)
gen_input = torch.cat([mri.to(device), fake_noise], 1)
out = gen(gen_input)

In [ ]:
show_tensor_images(out, num_images = 4, size = (1, 256, 256))

In [ ]:
im = Image.open('Generated_Images/image_3.png')

In [ ]:
im